# DataFrames II - Datos Faltantes, ordenamiento y conteos

In [1]:
import polars as pl

## El método `fill_null`
- Polars utiliza `null` para representar un valor faltante.

In [2]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(5)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01
"""Diana Weaver""","""HR""","""diana.weaver@polars.io""",84672,5,2019-11-25
"""Sierra Ross""",null,"""sierra.ross@polars.io""",148601,7,2018-02-14


- El método `fill_null` reemplaza los valores faltantes aplicando una de varias estrategias.
- Podemos proporcionar un valor constante, una estrategia de reemplazo o una expresión de Polars.

In [3]:
employees.select(
    pl.col("department"),
    pl.col("department").fill_null("Intern").alias("department2"),
).head(5)

department,department2
str,str
"""CEO""","""CEO"""
"""Operations""","""Operations"""
null,"""Intern"""
"""HR""","""HR"""
null,"""Intern"""


- El parámetro `strategy` especifica si se debe usar el valor anterior o siguiente presente para reemplazar un valor faltante.

In [4]:
employees.head(6).select(
    pl.col("department"),
    pl.col("department").fill_null(strategy="forward").alias("department_forward"),
    pl.col("department").fill_null(strategy="backward").alias("department_backward"),
)

department,department_forward,department_backward
str,str,str
"""CEO""","""CEO""","""CEO"""
"""Operations""","""Operations""","""Operations"""
null,"""Operations""","""HR"""
"""HR""","""HR""","""HR"""
null,"""HR""","""Marketing"""
"""Marketing""","""Marketing""","""Marketing"""


- Seleccionemos las primeras 5 filas y luego agreguemos una columna `title` al `DataFrame`.
- Podemos usar el título como la mejor coincidencia para un valor faltante en la columna `department`.

In [6]:
employees_with_title = employees.head(5).with_columns(
    pl.Series(
        "title",
        [
            "CEO",
            "Warehouse Manager",
            "Frontend Developer",
            "Acquisition Lead",
            "Backend Developer",
        ],
    )
)
employees_with_title

name,department,email,salary,years_at_company,start_date,title
str,str,str,i64,i64,date,str
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14,"""CEO"""
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13,"""Warehouse Manager"""
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01,"""Frontend Developer"""
"""Diana Weaver""","""HR""","""diana.weaver@polars.io""",84672,5,2019-11-25,"""Acquisition Lead"""
"""Sierra Ross""",null,"""sierra.ross@polars.io""",148601,7,2018-02-14,"""Backend Developer"""


In [7]:
employees_with_title.select(
    pl.col("department"),
    pl.col("title"),
    pl.col("department").fill_null(pl.col("title")).alias("department_updated")
)

department,title,department_updated
str,str,str
"""CEO""","""CEO""","""CEO"""
"""Operations""","""Warehouse Manager""","""Operations"""
null,"""Frontend Developer""","""Frontend Developer"""
"""HR""","""Acquisition Lead""","""HR"""
null,"""Backend Developer""","""Backend Developer"""


### Lectura adicional
- https://docs.pola.rs/user-guide/expressions/missing-data/#filling-missing-data
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.fill_null.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.fill_null.html

## Interpolación lineal con `interpolate`
- La interpolación reemplaza los valores faltantes usando interpolación lineal.


In [8]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


In [9]:
bonus_df = employees.head(6).select(
    pl.col("name")
    , pl.Series("bonus", [10000, None, 20000, None, None, 50000])
)
bonus_df

name,bonus
str,i64
"""Nicholas Maldonado""",10000
"""Michael Fletcher""",null
"""Jeffrey Tanner""",20000
"""Diana Weaver""",null
"""Sierra Ross""",null
"""Melissa Page""",50000


In [10]:
bonus_df.with_columns(pl.col("bonus").interpolate().alias("bonus_updated"))

name,bonus,bonus_updated
str,i64,f64
"""Nicholas Maldonado""",10000,10000.0
"""Michael Fletcher""",null,15000.0
"""Jeffrey Tanner""",20000,20000.0
"""Diana Weaver""",null,30000.0
"""Sierra Ross""",null,40000.0
"""Melissa Page""",50000,50000.0


### Lectura adicional
- https://docs.pola.rs/user-guide/expressions/missing-data/#fill-with-interpolation
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.interpolate.html

## Eliminación de datos faltantes con `drop_nulls`
- El método `drop_nulls` en una expresión elimina los valores `null` de la columna objetivo.
- Si hay valores faltantes, la nueva columna será más corta que la original.
- El método `with_columns` adjunta nuevas columnas al final del `DataFrame` y espera columnas de igual longitud.

In [11]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(5)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01
"""Diana Weaver""","""HR""","""diana.weaver@polars.io""",84672,5,2019-11-25
"""Sierra Ross""",null,"""sierra.ross@polars.io""",148601,7,2018-02-14


In [14]:
employees = employees.head(5).with_columns(pl.lit(None).alias("email"))
employees

name,department,email,salary,years_at_company,start_date
str,str,null,i64,i64,date
"""Nicholas Maldonado""","""CEO""",null,250000,9,2016-07-14
"""Michael Fletcher""","""Operations""",null,96540,9,2016-02-13
"""Jeffrey Tanner""",null,null,126489,10,2015-03-01
"""Diana Weaver""","""HR""",null,84672,5,2019-11-25
"""Sierra Ross""",null,null,148601,7,2018-02-14


- El método `drop_nulls` en un `DataFrame` elimina las filas que tienen uno o más valores `null`.
- El parámetro `subset` limita las columnas que Polars usa para identificar valores `null`.
- `pl.lit(None)` crear una columna con el mismo valor literal.

In [15]:
employees.drop_nulls()

name,department,email,salary,years_at_company,start_date
str,str,null,i64,i64,date


In [16]:
employees.drop_nulls(subset=["department"])

name,department,email,salary,years_at_company,start_date
str,str,null,i64,i64,date
"""Nicholas Maldonado""","""CEO""",null,250000,9,2016-07-14
"""Michael Fletcher""","""Operations""",null,96540,9,2016-02-13
"""Diana Weaver""","""HR""",null,84672,5,2019-11-25


In [17]:
employees.drop_nulls(subset=["department", "email"])

name,department,email,salary,years_at_company,start_date
str,str,null,i64,i64,date


In [22]:
# employees.select(
#     pl.col("department")
# )

In [23]:
# employees.select(
#     pl.col("department").drop_nulls().alias("department_not_nulls")
#     )

In [19]:
# Esto funcionaría?
# employees.select(
#     pl.col("department")
#     , pl.col("department").drop_nulls().alias("department_not_nulls")
#     )

### Lectura adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.drop_nulls.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop_nulls.html

## Ordenamiento por una sola columna
- El ordenamiento cambia el orden de las filas basándose en los valores de una o más columnas.
- El número de filas en el `DataFrame` permanece igual en el resultado ordenado.

In [24]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


- El orden predeterminado es ascendente (de menor a mayor, alfabético, de más antiguo a más reciente).
- Polars tiene un parámetro `descending` que por defecto es `False`.

- Un ordenamiento ascendente ordena las fechas de la más antigua a la más reciente.

In [25]:
employees.sort(pl.col("start_date"))

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Terry Walls""",null,"""terry.walls@polars.io""",89421,10,2014-07-24
"""Jeremy Harris""","""HR""","""jeremy.harris@polars.io""",74442,10,2014-07-25
"""Kylie Clarke""","""HR""","""kylie.clarke@polars.io""",66997,10,2014-08-01
"""Keith Gross""","""Marketing""","""keith.gross@polars.io""",128607,10,2014-08-11
"""Paul Guerrero""","""Engineering""","""paul.guerrero@polars.io""",184788,10,2014-08-15
…,…,…,…,…,…
"""Reginald Wallace""","""Finance""","""reginald.wallace@polars.io""",142361,0,2025-06-29
"""Isaiah Smith""","""HR""","""isaiah.smith@polars.io""",86023,0,2025-06-29
"""Carrie Montoya""","""Engineering""","""carrie.montoya@polars.io""",145307,0,2025-07-04


- Pasa `True` al parámetro `descending` para ordenar en orden descendente.

In [26]:
employees.sort(pl.col("salary"), descending=True)
# employees.sort(pl.col("name"), descending=True)
# employees.sort(pl.col("start_date"), descending=True)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Jack Tanner""","""Engineering""","""jack.tanner@polars.io""",199503,1,2024-02-23
"""Shawn Gray""","""Engineering""","""shawn.gray@polars.io""",199381,2,2022-12-15
"""Shannon Klein""","""Finance""","""shannon.klein@polars.io""",199260,7,2017-07-19
"""James Wilson""","""Finance""","""james.wilson@polars.io""",199257,2,2023-01-06
…,…,…,…,…,…
"""Brenda Lopez""",null,"""brenda.lopez@polars.io""",55304,4,2021-04-19
"""Cristina Williams""",null,"""cristina.williams@polars.io""",55242,8,2016-07-21
"""Aaron Morgan""",null,"""aaron.morgan@polars.io""",55078,10,2014-08-21


- El parámetro `nulls_last` puede forzar que los valores `null` (faltantes) aparezcan al final del ordenamiento.

In [29]:
# employees.sort(pl.col("department"))

employees.sort(pl.col("department"), nulls_last=True)

# employees.sort(pl.col("department"), nulls_last=False, descending=True)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Brian Cooper""","""Engineering""","""brian.cooper@polars.io""",156239,10,2015-02-03
"""Mary Long""","""Engineering""","""mary.long@polars.io""",173633,9,2015-11-27
"""Amanda Rodriguez""","""Engineering""","""amanda.rodriguez@polars.io""",127603,0,2024-11-27
"""Laura Rosales""","""Engineering""","""laura.rosales@polars.io""",147444,2,2022-09-25
…,…,…,…,…,…
"""Amber Smith""",null,"""amber.smith@polars.io""",88525,0,2024-08-09
"""Jennifer Murphy""",null,"""jennifer.murphy@polars.io""",79626,1,2024-07-13
"""James Bryant""",null,"""james.bryant@polars.io""",85285,9,2016-05-09


In [ ]:
# CUIDADO! con esto, solamente se ordenará la columna perdiendo
# la conexión original entre el empleado y su salario
# employees.with_columns(pl.col("salary").sort())

# employees.with_columns(pl.col("salary").sort(descending=True))

# Esto si ordena las filas del DF
# employees.sort(pl.col("salary"))

### Lectura adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.sort.html

## Ordenamiento por múltiples columnas I
- Pasa múltiples cadenas al método `sort` para ordenar por múltiples columnas.
- Polars aplicará un ordenamiento ascendente uniforme a cada columna por defecto.

In [30]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


- Polars colocará los valores `null` primero, luego ordenará por departamento y después ordenará por nombre dentro de cada departamento.

In [31]:
employees.sort(pl.col("department", "name"), nulls_last=True)

# employees.sort("department", "name", nulls_last=True)

# employees.sort(["department", "name"], nulls_last=True)

# employees.sort(pl.col("department"), pl.col("name"), nulls_last=True)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Amanda Carter""","""Engineering""","""amanda.carter@polars.io""",175965,10,2015-02-04
"""Amanda Meyer""","""Engineering""","""amanda.meyer@polars.io""",128286,1,2024-02-27
"""Amanda Rodriguez""","""Engineering""","""amanda.rodriguez@polars.io""",127603,0,2024-11-27
"""Amy Pham""","""Engineering""","""amy.pham@polars.io""",175067,7,2017-08-22
…,…,…,…,…,…
"""Troy Allen""",null,"""troy.allen@polars.io""",55432,8,2017-05-08
"""Valerie Rivera""",null,"""valerie.rivera@polars.io""",73413,4,2021-03-08
"""Veronica Gutierrez""",null,"""veronica.gutierrez@polars.io""",65010,6,2019-02-09


## Ordenamiento por múltiples columnas II
- Polars ordena cada columna en orden ascendente por defecto.
- Pasa al parámetro `descending` una lista de booleanos para personalizar el orden de clasificación por columna.
- El número de entradas en la lista debe coincidir con el número de columnas.

In [32]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


- Proporciona tanto `True` como `False` para ordenar en diferente orden por columna.
- `[True, False]` ordena `department` en orden descendente y `name` en orden ascendente dentro de cada departamento.
- `[False, True]` ordena `department` en orden ascendente y `name` en orden descendente dentro de cada departamento.

In [33]:
employees.sort(pl.col("department"), pl.col("name"), descending=[True, False])

# employees.sort("department", "name", descending=[False, True])

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Aaron Morgan""",null,"""aaron.morgan@polars.io""",55078,10,2014-08-21
"""Adrian Graham""",null,"""adrian.graham@polars.io""",68575,10,2015-06-08
"""Alicia Mueller""",null,"""alicia.mueller@polars.io""",74673,2,2022-08-15
"""Alyssa Harper DDS""",null,"""alyssa.dds@polars.io""",69409,2,2022-12-29
"""Amber Smith""",null,"""amber.smith@polars.io""",88525,0,2024-08-09
…,…,…,…,…,…
"""Timothy Williams""","""Engineering""","""timothy.williams@polars.io""",184248,5,2020-04-12
"""Todd Bates""","""Engineering""","""todd.bates@polars.io""",173707,4,2021-04-26
"""Todd Torres""","""Engineering""","""todd.torres@polars.io""",144105,7,2017-12-13


## Caracteres vs Bytes

In [34]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


- Anteriormente, presentamos el atributo `name` para métodos que trabajan con nombres de columnas.
- Como siempre, los métodos devuelven una nueva expresión que podemos aplicar en un contexto específico de Polars.

In [35]:
employees.select(pl.all().name.to_uppercase()).head(1)

NAME,DEPARTMENT,EMAIL,SALARY,YEARS_AT_COMPANY,START_DATE
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14


- El atributo `str` contiene métodos para manipulaciones de cadenas de texto en las columnas.
- Los métodos `to_lowercase` y `to_uppercase` convierten a minúsculas o mayúsculas los valores de una columna.

In [36]:
employees.select(
    pl.col("name").str.to_lowercase().alias("lower"),
    pl.col("name").str.to_uppercase().alias("upper"),
)

lower,upper
str,str
"""nicholas maldonado""","""NICHOLAS MALDONADO"""
"""michael fletcher""","""MICHAEL FLETCHER"""
"""jeffrey tanner""","""JEFFREY TANNER"""
"""diana weaver""","""DIANA WEAVER"""
"""sierra ross""","""SIERRA ROSS"""
…,…
"""james bryant""","""JAMES BRYANT"""
"""patricia vazquez""","""PATRICIA VAZQUEZ"""
"""katie clay""","""KATIE CLAY"""


- El método `str.len_chars` devuelve la cantidad de caracteres en una cadena.
- El método complementario `str.len_bytes` cuenta los bytes en una cadena.
- Un carácter alfabético en inglés ocupa un byte en memoria.
- La relación 1 a 1 no siempre es verdadera. Un emoji como 🍕 es 1 carácter pero ocupa 4 bytes.

In [38]:
pl.DataFrame({"foods": ["pizza", "🍕"]})

foods
str
"""pizza"""
"""🍕"""


In [39]:
pl.DataFrame({"foods": ["pizza", "🍕"]}).with_columns(
    pl.col("foods").str.len_chars().alias("length_in_chars"),
    pl.col("foods").str.len_bytes().alias("length_in_bytes"),
)

foods,length_in_chars,length_in_bytes
str,u32,u32
"""pizza""",5,5
"""🍕""",1,4


### Lectura adicional
- https://docs.pola.rs/user-guide/expressions/strings/#the-string-namespace
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.to_lowercase.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.to_uppercase.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.len_bytes.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.len_chars.html

## Ordenamiento con expresiones
- Podemos pasar una expresión al método `sort` para personalizar el ordenamiento.

In [40]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


In [41]:
employees.select(pl.col("name").str.len_chars().alias("name_len_chars"))

name_len_chars
u32
18
16
14
12
11
…
12
16
10


In [43]:
name_lengths = pl.col("name").str.len_chars()

- Podemos usar la expresión como base de un ordenamiento personalizado.
- **Ordenemos las filas según la longitud de los nombres de los empleados**.
- Las expresiones personalizadas se pueden combinar con expresiones de columnas simples.

In [44]:
employees.sort(name_lengths)

# employees.sort(name_lengths, descending=True)

# Dentro de cada department ordena por la longitud de los nombres de los empleados
# employees.sort("department", name_lengths, nulls_last=True)

# employees.sort("department", name_lengths, descending=[False, True])

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Jane Lee""","""Marketing""","""jane.lee@polars.io""",134795,0,2025-06-22
"""Amy Rice""","""Finance""","""amy.rice@polars.io""",191529,4,2020-08-27
"""Joe Cruz""","""Engineering""","""joe.cruz@polars.io""",136805,10,2014-08-18
"""Kim Cook""","""HR""","""kim.cook@polars.io""",95014,2,2022-09-28
"""Eric Fox""","""Engineering""","""eric.fox@polars.io""",147294,9,2015-12-11
…,…,…,…,…,…
"""Mrs. Patricia Scott PhD""","""Finance""","""mrs..phd@polars.io""",158684,3,2021-10-13
"""Dr. Cheryl Hernandez MD""","""Sales""","""dr..md@polars.io""",68551,8,2017-01-15
"""Mrs. Jennifer Price DDS""","""Engineering""","""mrs..dds@polars.io""",185325,5,2019-11-23


## Los métodos `top_k` y `bottom_k`
- El método `top_k` extrae una cantidad especificada de los valores más grandes/máximos.
- El método `bottom_k` extrae una cantidad especificada de los valores más pequeños/mínimos.

In [45]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


In [46]:
employees.select(
    pl.col("salary").top_k(3).alias("top_3_salaries"),
    pl.col("salary").bottom_k(3).alias("bottom_3_salaries"),
)

top_3_salaries,bottom_3_salaries
i64,i64
250000,55011
199503,55012
199381,55078


### Lectura adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.top_k.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.bottom_k.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.top_k.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.bottom_k.html

## Conteo y extracción de valores únicos
- El método `n_unique` cuenta la cantidad exacta de valores únicos en una columna.
- Polars incluye un valor `null`/faltante en el conteo.

In [47]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(3)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13
"""Jeffrey Tanner""",null,"""jeffrey.tanner@polars.io""",126489,10,2015-03-01


In [48]:
employees.select(pl.col("department").n_unique())

# employees.select(pl.col("department", "email").n_unique())

# employees.select(pl.all().n_unique())

department
u32
8


- El método `unique` devuelve los valores únicos de la columna especificada.
- Cada valor único se lista una sola vez.

In [49]:
employees.select(pl.col("department").unique())

# employees.select(pl.col("department", "email").unique())

department
str
"""Marketing"""
"""CEO"""
"""Finance"""
"""Operations"""
"""Sales"""
"""Engineering"""
null
"""HR"""


### Lectura adicional
- https://docs.pola.rs/user-guide/expressions/basic-operations/#counting-unique-values
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.unique.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.approx_n_unique.html

## El método value_counts
- El método `value_counts` cuenta el número de ocurrencias de cada valor único.
- El método `value_counts` devuelve una columna de structs.
- Un struct es una estructura de datos comparable a un diccionario de Python. Consiste en pares clave-valor.
- El struct de cada fila contiene dos datos: el nombre del `department` y su conteo.

In [50]:
employees = pl.read_csv("employees.csv", try_parse_dates=True)
employees.head(2)

name,department,email,salary,years_at_company,start_date
str,str,str,i64,i64,date
"""Nicholas Maldonado""","""CEO""","""nicholas.maldonado@polars.io""",250000,9,2016-07-14
"""Michael Fletcher""","""Operations""","""michael.fletcher@polars.io""",96540,9,2016-02-13


In [51]:
employees.select(pl.col("department").value_counts())

department
struct[2]
"{""Engineering"",139}"
"{""Marketing"",146}"
"{""CEO"",1}"
"{""Operations"",136}"
"{""Finance"",144}"
"{""Sales"",129}"
"{null,155}"
"{""HR"",150}"


- El método `unnest` en un `DataFrame` extrae los valores del struct de cada fila en columnas separadas.
- Los pares clave-valor del struct están "anidados" dentro del struct — esto los "desanida".

In [52]:
employees.select(pl.col("department").value_counts()).unnest(pl.col("department"))

department,count
str,u32
"""Operations""",136
null,155
"""Sales""",129
"""Engineering""",139
"""Finance""",144
"""Marketing""",146
"""HR""",150
"""CEO""",1


- Pasa `True` al parámetro `sort` para ordenar por el valor con mayor cantidad de ocurrencias.

In [ ]:
employees.select(pl.col("department").value_counts(sort=True))

department
struct[2]
"{null,155}"
"{""HR"",150}"
"{""Marketing"",146}"
"{""Finance"",144}"
"{""Engineering"",139}"
"{""Operations"",136}"
"{""Sales"",129}"
"{""CEO"",1}"


- Establece `normalize` en `True` para ver los porcentajes relativos de cada valor único.

In [54]:
employees.select(pl.col("department").value_counts(sort=True, normalize=True))

department
struct[2]
"{null,0.155}"
"{""HR"",0.15}"
"{""Marketing"",0.146}"
"{""Finance"",0.144}"
"{""Engineering"",0.139}"
"{""Operations"",0.136}"
"{""Sales"",0.129}"
"{""CEO"",0.001}"


- Observa que el nombre de la columna es `proportion` en lugar de `count`.
- El nombre de la columna proviene directamente de la clave dentro del struct.

In [53]:
employees.select(pl.col("department").value_counts(sort=True, normalize=True)).unnest(
    pl.col("department")
)

department,proportion
str,f64
null,0.155
"""HR""",0.15
"""Marketing""",0.146
"""Finance""",0.144
"""Engineering""",0.139
"""Operations""",0.136
"""Sales""",0.129
"""CEO""",0.001


### Lectura adicional
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.value_counts.html

---
# Ejercicios de práctica

In [ ]:
# Ejecuta esta celda para crear el DataFrame de práctica
import polars as pl

ventas = pl.DataFrame({
    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Laptop", "Mouse", "Auriculares", "Teclado", "Monitor", "Laptop"],
    "categoria": ["Electrónica", None, "Accesorios", "Electrónica", "Electrónica", "Accesorios", None, "Accesorios", None, "Electrónica"],
    "precio": [1200, 25, None, 350, 1150, 30, 80, None, 400, 1300],
    "cantidad": [5, None, 20, 8, 3, None, 15, 25, 6, 2],
    "vendedor": ["Ana", "Luis", "Ana", "María", "Luis", "Ana", "María", "Luis", "Ana", "María"],
})
ventas

producto,categoria,precio,cantidad,vendedor
str,str,i64,i64,str
"""Laptop""","""Electrónica""",1200,5,"""Ana"""
"""Mouse""",null,25,null,"""Luis"""
"""Teclado""","""Accesorios""",null,20,"""Ana"""
"""Monitor""","""Electrónica""",350,8,"""María"""
"""Laptop""","""Electrónica""",1150,3,"""Luis"""
"""Mouse""","""Accesorios""",30,null,"""Ana"""
"""Auriculares""",null,80,15,"""María"""
"""Teclado""","""Accesorios""",null,25,"""Luis"""
"""Monitor""",null,400,6,"""Ana"""


### Ejercicio 1
Reemplaza los valores `null` de la columna `categoria` con el texto `"Sin categoría"` y los valores `null` de la columna `precio` usando la estrategia `forward`. Muestra el resultado con `select`.

### Ejercicio 2
Ordena el DataFrame `ventas` por `categoria` en orden ascendente y por `precio` en orden descendente. Los valores `null` deben aparecer al final.

### Ejercicio 3
Usa `value_counts` sobre la columna `vendedor` para contar cuántas ventas realizó cada vendedor. Desanida el resultado con `unnest` y ordena de mayor a menor cantidad de ventas.

### Ejercicio 4
Convierte los nombres de los productos a mayúsculas y agrega una columna `longitud_producto` con la cantidad de caracteres de cada nombre. Luego ordena el DataFrame por `longitud_producto` en orden descendente.

### Ejercicio 5
Elimina las filas con valores nulos en la columna `precio`. Luego, extrae los 3 precios más altos usando `top_k` y los 3 más bajos usando `bottom_k`. Agrega una columna `ranking_precio` que clasifique los productos por precio de mayor a menor usando `rank`.

**Nota:** Se debe averiguar qué hacer el método `rank`.